# Lab 6: Logistic regression from scratch

Scale -> initialize -> sigmoid -> error -> gradients -> update -> predict -> metrics.
NumPy only. This follows LogisticRegFlow.png and ClassificationMetrics.png.
There is no original Lab 6 notebook in the workspace.

The eight-row toy dataset makes arithmetic checks manageable. Its reported accuracy
is training accuracy, not evidence of performance on unseen data. If the question
requires a test split, split first and normalize using training statistics.

In [1]:
import numpy as np


## 1. A small dataset (hours studied, hours slept -> pass=1/fail=0)

In [2]:
X = np.array([[2,9],[1,5],[3,6],[4,8],[5,7],[6,9],[7,5],[8,8]], dtype=float)
y = np.array([0,0,0,0,1,1,1,1])

# normalise (helps gradient descent)
mu, sd = X.mean(0), X.std(0)
Xn = (X - mu) / sd
print("X shape:", Xn.shape, "| labels:", y)


X shape: (8, 2) | labels: [0 0 0 0 1 1 1 1]


## 2. Sigmoid

In [3]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


## 3. Train with Gradient Descent

In [4]:
def train(X, y, lr=0.5, n_iters=20000):
    m, n = X.shape
    w = np.zeros(n)
    b = 0.0             # 2. init params
    for _ in range(n_iters):
        yhat = sigmoid(X @ w + b)        # 3-4. z then sigmoid
        error = yhat - y                 # 5. error
        dw = (1/m) * X.T @ error         # 6. gradients
        db = (1/m) * np.sum(error)
        w -= lr * dw                     # 7. update
        b -= lr * db
    return w, b

w, b = train(Xn, y)
print("weights:", np.round(w,3), "| bias:", round(b,3))


weights: [18.488 -4.977] | bias: 1.191


## 4. Predict (probability -> label at 0.5)

In [5]:
proba = sigmoid(Xn @ w + b)
y_pred = (proba >= 0.5).astype(int)
print("probabilities:", np.round(proba,2))
print("predicted    :", y_pred)
print("actual       :", y)


probabilities: [0. 0. 0. 0. 1. 1. 1. 1.]
predicted    : [0 0 0 0 1 1 1 1]
actual       : [0 0 0 0 1 1 1 1]


## 5. Classification metrics (from scratch)

In [6]:
def metrics(yt, yp):
    TP = np.sum((yt == 1) & (yp == 1))
    TN = np.sum((yt == 0) & (yp == 0))
    FP = np.sum((yt == 0) & (yp == 1))
    FN = np.sum((yt == 1) & (yp == 0))
    acc = (TP + TN) / len(yt)
    prec = TP / (TP + FP) if TP + FP else 0
    rec = TP / (TP + FN) if TP + FN else 0
    spec = TN / (TN + FP) if TN + FP else 0
    f1 = 2 * TP / (2 * TP + FP + FN) if 2 * TP + FP + FN else 0
    fpr = FP / (FP + TN) if FP + TN else 0
    fnr = FN / (FN + TP) if FN + TP else 0
    print("Rows=actual, columns=predicted, label order [1, 0]:")
    print(np.array([[TP, FN], [FP, TN]]))
    print("Accuracy / Error rate:", acc, (FP + FN) / len(yt))
    print("Precision / Recall / Specificity / F1:", prec, rec, spec, f1)
    print("FPR / FNR / Balanced accuracy:", fpr, fnr, (rec + spec) / 2)

metrics(y, y_pred)


Rows=actual, columns=predicted, label order [1, 0]:
[[4 0]
 [0 4]]
Accuracy / Error rate: 1.0 0.0
Precision / Recall / Specificity / F1: 1.0 1.0 1.0 1.0
FPR / FNR / Balanced accuracy: 0.0 0.0 1.0


## 6. Practice check: the first gradient-descent step

At zero weights and bias, all probabilities are 0.5. The gradient is `dw`.
The new weights are `w - lr*dw`. Do not confuse the gradient with the update.

In [7]:
w0 = np.zeros(Xn.shape[1])
b0 = 0.0
p0 = sigmoid(Xn @ w0 + b0)
dw0 = Xn.T @ (p0 - y) / len(y)
db0 = np.mean(p0 - y)
print("Initial probabilities:", p0)
print("Gradients dw, db:", dw0.round(4), round(db0, 4))
print("New weights, bias:", (w0 - 0.5 * dw0).round(4), b0 - 0.5 * db0)


Initial probabilities: [0.5 0.5 0.5 0.5 0.5 0.5 0.5 0.5]
Gradients dw, db: [-0.4364 -0.0407] 0.0
New weights, bias: [0.2182 0.0203] 0.0


## 7. Predict for new data

Use the training mean and standard deviation again.

In [8]:
X_new = np.array([[2, 7], [7, 7]], dtype=float)
X_new_n = (X_new - mu) / sd
new_proba = sigmoid(X_new_n @ w + b)
print("New probabilities:", new_proba.round(3))
print("New labels:", (new_proba >= 0.5).astype(int))


New probabilities: [0. 1.]
New labels: [0 1]
